In [1]:
import requests
import datetime
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
INPUT_DATASET_PATH = '/home/voamorim/Documents/UFSJ/6o_periodo/DataMining/ForkNathan/lei_felca/data/preprocessed_title_description.csv'

In [3]:
# define cores de saida no terminal 
class bcolors:
    HEADER = '\033[95m'
    OKBLUE = '\033[94m'
    OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'
    WARNING = '\033[93m'
    FAIL = '\033[91m'
    ENDC = '\033[0m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'

# imprime o momento exato no terminal
def print_time():
    date = datetime.datetime.now()
    message = f'[{str(date.month)}/{str(date.day)}/{str(date.year)} {str(date.strftime("%H:%M"))}] - '
    print(message, end="")

# acessa as urls uma a uma. retorna a lista final com as urls com os resultados
# apos os redirecionamentos e as urls que nao foram possiveis de serem acessadas
def access_urls(urls: list) -> list:
    final_urls = list()
    failed_urls = list()

    for url in urls:
        try:
            response = requests.head(url, allow_redirects=True, timeout=10)
            final_url = response.url
            final_urls.append(final_url)

            print_time() 
            print(f'{bcolors.OKGREEN}URL accessed successfully. Final URL: {final_url}{bcolors.ENDC}')
        except Exception as e:
            print_time() 
            print(f'{bcolors.FAIL}An unexpected error occurred while accessing the url. Error message: {e}{bcolors.ENDC}')
            failed_urls.append(url)

    return final_urls, failed_urls

# cosntroi a url do youtube a ser acessada com base no id
# forca acessar um video shorts. youtube automaticamente redireciona para outra
# pagina caso o video nao seja um shorts
def buildUrlFromId(id:str) ->str:
    url = f"https://www.youtube.com/shorts/{id}"
    return url

# divide a lista de urls de entrada em partes menores 
def divideUrlsInChunks(urls:list):
    lots = [] 
    for i in range(0, len(urls), 100):
        j = i + 100
        lot = urls[i:min(j, len(urls))]
        lots.append(lot)
    return lots

if __name__ == '__main__':
    df = pd.read_csv(INPUT_DATASET_PATH)
    video_ids = df['videoId'].tolist()

    # monta todas as urls a partir do id e as divide em partes menores 
    urls = [buildUrlFromId(id) for id in video_ids]
    chunks = divideUrlsInChunks(urls)

    # chama a funcao de fazer as requisicoes para cada um dos chunks. 
    # usa 10 threads para agilizar o processo
    final_urls = []
    failed_urls = []
    with ThreadPoolExecutor(max_workers=10) as executor:
        f = {executor.submit(access_urls, chunk): chunk for chunk in chunks}

        for future in as_completed(f):
            try:
                final, failed = future.result()
                final_urls.extend(final)
                failed_urls.extend(failed)
            except Exception as e:
                print(f'{bcolors.FAIL}An exception has occurred: {e}{bcolors.ENDC}')
    
    # salva urls que foram acessadas com sucesso 
    with open("../shorts/redirected.txt", mode='w') as fout:
        for url in final_urls:
            fout.write(url + '\n')

    if len(failed_urls):
        # salva as urls inacessiveis 
        with open("../shorts/failed.txt", mode='w') as ferror:
            for url in failed_urls:
                ferror.write(url + '\n')

[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/watch?v=XS3LqC9HbG0
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/VsAvkG0lp60
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/5pZugs2Yncs
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/qWT00zjS8bo
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/1Wy-bekWX9Q
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/4RNuCeLeX0Y
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/7b7hoZ_UWYk
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/w22lrhGXy_I
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/frmsac2kFME
[4/12/2026 14:58] - URL accessed successfully. Final URL: https://www.youtube.com/shorts/v